# Encyclopedia Editing Tutorial

**Date:** February 21, 2026 (system date)  
**Purpose:** Learn how to edit encyclopedias using Python

This tutorial demonstrates how to:
- Delete entries (soft and hard delete)
- Hide/show entries
- Mark entries as needing editing
- Add entries from Wikipedia
- Merge encyclopedias

All operations save outputs to `temp/` for inspection.

## Setup

First, import required modules and set up paths.

In [1]:
from pathlib import Path
from encyclopedia.core.encyclopedia import AmiEncyclopedia
from encyclopedia.utils.resources import Resources

# Set up output directory for this tutorial
tutorial_output_dir = Path(Resources.TEMP_DIR, "tutorials", "editing")
tutorial_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {tutorial_output_dir}")

<root id="root">
        <section>
            <item id="item1"/>
            <item/>
            <item/>
        </section>
        <section>
            <item/>
            <item/>
        </section>
    </root>
Output directory: /Users/pm286/workspace/encyclopedia/temp/tutorials/editing


## Load an Encyclopedia

Load an existing encyclopedia file. If you don't have one, create one first using `create_encyclopedia_from_wordlist.py`.

In [2]:
# Load encyclopedia
# Replace with your encyclopedia file path
print(f"dir {Path()}")
encyclopedia_file = Path("my_encyclopedia.html")

if not encyclopedia_file.exists():
    print(f"⚠ Encyclopedia file not found: {encyclopedia_file}")
    print("Please create an encyclopedia first or update the path above.")
else:
    encyclopedia = AmiEncyclopedia()
    encyclopedia.create_from_html_file(encyclopedia_file)
    
    print(f"✓ Loaded encyclopedia: {encyclopedia.title}")
    print(f"  Total entries: {len(encyclopedia.entries)}")
    print(f"  Version: {encyclopedia.metadata.get('version')}")

dir .
⚠ Encyclopedia file not found: my_encyclopedia.html
Please create an encyclopedia first or update the path above.


## 1. Delete Entries

### Soft Delete (Recoverable)

Soft delete marks an entry as deleted in metadata but keeps it in the entries list. This allows recovery.

In [3]:
# Get first entry as example
if len(encyclopedia.entries) > 0:
    entry = encyclopedia.entries[0]
    entry_id = encyclopedia._generate_entry_id_from_entry(entry, 0)
    term = entry.get('term', 'Unknown')
    
    print(f"Example entry: {term} (ID: {entry_id})")
    print(f"\nSoft deleting entry...")
    
    # Soft delete
    result = encyclopedia.delete_entry(entry_id, soft=True)
    
    if result:
        print(f"✓ Entry soft deleted")
        print(f"  Entry still in entries list: {len(encyclopedia.entries)} entries")
        print(f"  Deleted entries count: {len(encyclopedia.get_deleted_entries())}")
    else:
        print(f"✗ Failed to delete entry")

NameError: name 'encyclopedia' is not defined

### Hard Delete (Permanent)

Hard delete removes the entry from the entries list. It's still recoverable from metadata for 30 days.

In [ ]:
# Hard delete example (if you have a second entry)
if len(encyclopedia.entries) > 1:
    entry = encyclopedia.entries[1]
    entry_id = encyclopedia._generate_entry_id_from_entry(entry, 1)
    term = entry.get('term', 'Unknown')
    initial_count = len(encyclopedia.entries)
    
    print(f"Hard deleting entry: {term} (ID: {entry_id})")
    
    # Hard delete
    result = encyclopedia.delete_entry(entry_id, soft=False)
    
    if result:
        print(f"✓ Entry hard deleted")
        print(f"  Entries before: {initial_count}")
        print(f"  Entries after: {len(encyclopedia.entries)}")
        print(f"  Deleted entries count: {len(encyclopedia.get_deleted_entries())}")

### Restore Deleted Entry

Restore a deleted entry from metadata.

In [ ]:
# List deleted entries
deleted = encyclopedia.get_deleted_entries()

if deleted:
    print(f"Deleted entries ({len(deleted)}):")
    for de in deleted:
        print(f"  - {de.get('term')} (ID: {de.get('entry_id')}, soft: {de.get('soft')})")
    
    # Restore first deleted entry
    first_deleted = deleted[0]
    entry_id = first_deleted.get('entry_id')
    
    print(f"\nRestoring entry: {entry_id}")
    result = encyclopedia.restore_entry(entry_id)
    
    if result:
        print(f"✓ Entry restored")
        print(f"  Entries now: {len(encyclopedia.entries)}")
        print(f"  Deleted entries: {len(encyclopedia.get_deleted_entries())}")
else:
    print("No deleted entries to restore")

## 2. Hide/Show Entries

Hide entries temporarily without deleting them. Hidden entries remain in the data but are marked in HTML.

In [ ]:
# Hide an entry
if len(encyclopedia.entries) > 0:
    entry = encyclopedia.entries[0]
    entry_id = encyclopedia._generate_entry_id_from_entry(entry, 0)
    term = entry.get('term', 'Unknown')
    
    print(f"Hiding entry: {term} (ID: {entry_id})")
    result = encyclopedia.hide_entry(entry_id)
    
    if result:
        print(f"✓ Entry hidden")
        print(f"  Hidden entries: {len(encyclopedia.get_hidden_entries())}")
    
    # Show entry
    print(f"\nShowing entry: {entry_id}")
    result = encyclopedia.show_entry(entry_id)
    
    if result:
        print(f"✓ Entry shown")
        print(f"  Hidden entries: {len(encyclopedia.get_hidden_entries())}")

## 3. Mark Entries as Needing Editing

Flag entries that need attention (e.g., disambiguation pages, incomplete entries).

In [ ]:
# Mark entry as needing editing
if len(encyclopedia.entries) > 0:
    entry = encyclopedia.entries[0]
    entry_id = encyclopedia._generate_entry_id_from_entry(entry, 0)
    term = entry.get('term', 'Unknown')
    
    print(f"Marking entry: {term} (ID: {entry_id})")
    print(f"  Reason: disambiguation")
    print(f"  Notes: This is a disambiguation page")
    
    result = encyclopedia.mark_entry_needs_editing(
        entry_id, 
        reason="disambiguation",
        notes="This is a disambiguation page"
    )
    
    if result:
        print(f"✓ Entry marked as needing editing")
        
        # Check entry
        needs_editing = entry.get('needs_editing', {})
        print(f"  Flag: {needs_editing.get('flag')}")
        print(f"  Reason: {needs_editing.get('reason')}")
        print(f"  Notes: {needs_editing.get('notes')}")
    
    # Get all entries needing editing
    needing_editing = encyclopedia.get_entries_needing_editing()
    print(f"\nTotal entries needing editing: {len(needing_editing)}")
    
    # Filter by reason
    disambiguation_entries = encyclopedia.get_entries_needing_editing(reason="disambiguation")
    print(f"Disambiguation entries: {len(disambiguation_entries)}")
    
    # Resolve editing flag
    print(f"\nResolving editing flag for: {entry_id}")
    result = encyclopedia.resolve_entry_editing(entry_id)
    if result:
        print(f"✓ Editing flag resolved")

## 4. Add Entries from Wikipedia

Search Wikipedia and add new entries, with automatic duplicate detection.

In [ ]:
# Add entry from Wikipedia
new_term = "quantum mechanics"
initial_count = len(encyclopedia.entries)

print(f"Adding entry from Wikipedia: {new_term}")
print(f"Current entries: {initial_count}")

result = encyclopedia.add_entry_from_wikipedia(new_term, check_duplicates=True)

if result.get('error'):
    print(f"✗ Error: {result['error']}")
elif result.get('is_duplicate'):
    print(f"✗ Duplicate detected!")
    existing = result.get('existing_entry')
    match_type = result.get('match_type')
    if existing:
        print(f"  Existing entry: {existing.get('term')}")
        print(f"  Match type: {match_type}")
elif result.get('added'):
    entry = result.get('entry')
    print(f"✓ Entry added successfully!")
    print(f"  Term: {entry.get('term')}")
    print(f"  Wikipedia URL: {entry.get('wikipedia_url')}")
    print(f"  Wikidata ID: {entry.get('wikidata_id') or 'None'}")
    print(f"  New total entries: {len(encyclopedia.entries)}")
    
    # Check if marked as needing editing (e.g., disambiguation)
    entry_id = encyclopedia._generate_entry_id_from_entry(entry, len(encyclopedia.entries) - 1)
    needs_editing = entry.get('needs_editing', {})
    if needs_editing.get('flag'):
        print(f"  ⚠ Marked as needing editing: {needs_editing.get('reason')}")

### Check for Duplicates

Check if an entry would be a duplicate before adding.

In [ ]:
# Check for duplicate
test_entry = {
    'term': 'test term',
    'wikidata_id': '',
    'wikipedia_url': 'https://en.wikipedia.org/wiki/test_term'
}

is_duplicate, existing_entry, match_type = encyclopedia.check_duplicate_entry(test_entry)

if is_duplicate:
    print(f"✗ Duplicate found!")
    print(f"  Existing entry: {existing_entry.get('term')}")
    print(f"  Match type: {match_type}")
else:
    print(f"✓ No duplicate found")
    print(f"  Safe to add entry")

## 5. Merge Encyclopedias

Merge one encyclopedia into another, handling duplicates and conflicts.

In [ ]:
# Create a second encyclopedia for merging
source_encyclopedia = AmiEncyclopedia(title="Source Encyclopedia")

# Add some entries
source_entry1 = {
    'term': 'new term 1',
    'canonical_term': 'new term 1',
    'wikidata_id': '',
    'wikipedia_url': 'https://en.wikipedia.org/wiki/new_term_1',
    'description_html': '<p>New term 1 description</p>',
    'definition_html': 'New term 1 definition',
    'synonyms': []
}

source_entry2 = {
    'term': 'new term 2',
    'canonical_term': 'new term 2',
    'wikidata_id': '',
    'wikipedia_url': 'https://en.wikipedia.org/wiki/new_term_2',
    'description_html': '<p>New term 2 description</p>',
    'definition_html': 'New term 2 definition',
    'synonyms': []
}

source_encyclopedia.entries.append(source_entry1)
source_encyclopedia.entries.append(source_entry2)

print(f"Source encyclopedia: {len(source_encyclopedia.entries)} entries")
print(f"Target encyclopedia: {len(encyclopedia.entries)} entries")

# Merge (no conflict resolution - will use defaults)
print(f"\nMerging source into target...")
result = encyclopedia.merge_encyclopedia(source_encyclopedia, conflict_resolution=None)

print(f"✓ Merge completed!")
print(f"  Entries added: {result.get('entries_added', 0)}")
print(f"  Entries merged: {result.get('entries_merged', 0)}")
print(f"  Conflicts detected: {len(result.get('conflicts', []))}")
print(f"  Conflicts resolved: {len(result.get('resolved_conflicts', []))}")
print(f"  Final total entries: {len(encyclopedia.entries)}")

### Merge with Conflict Resolution

Resolve conflicts when merging encyclopedias.

In [ ]:
# Define conflict resolution strategy
# Options: "keep_target", "replace_with_source", "merge", "skip"
conflict_resolution = {
    'conflicting_term': 'replace_with_source',  # Replace target with source
    'another_term': 'merge'  # Merge entries
}

# Merge with conflict resolution
# result = encyclopedia.merge_encyclopedia(source_encyclopedia, conflict_resolution=conflict_resolution)

print("Conflict resolution strategies:")
print("  - keep_target: Keep existing entry (default)")
print("  - replace_with_source: Replace with source entry")
print("  - merge: Combine entries")
print("  - skip: Skip source entry")

## 6. Save Edited Encyclopedia

Save your edited encyclopedia. Version will be automatically bumped.

In [ ]:
# Save edited encyclopedia
output_file = Path(tutorial_output_dir, "edited_encyclopedia.html")

print(f"Saving to: {output_file}")
print(f"Version before save: {encyclopedia.metadata.get('version')}")

encyclopedia.save_wiki_normalized_html(output_file)

print(f"✓ Encyclopedia saved!")
print(f"Version after save: {encyclopedia.metadata.get('version')}")
print(f"Total entries: {len(encyclopedia.entries)}")
print(f"Hidden entries: {len(encyclopedia.get_hidden_entries())}")
print(f"Deleted entries: {len(encyclopedia.get_deleted_entries())}")
print(f"Entries needing editing: {len(encyclopedia.get_entries_needing_editing())}")

## Summary

You've learned how to:

1. ✅ **Delete entries** - Soft delete (recoverable) or hard delete (permanent)
2. ✅ **Hide/show entries** - Temporarily hide entries without deleting
3. ✅ **Mark entries for editing** - Flag entries that need attention
4. ✅ **Add entries from Wikipedia** - Search and add with duplicate detection
5. ✅ **Merge encyclopedias** - Combine two encyclopedias with conflict resolution
6. ✅ **Save changes** - All changes are persisted with version bumping

### Next Steps

- Use the command-line scripts in `Examples/` for batch operations
- Integrate editing into your workflow
- Use Streamlit browser for interactive editing (coming soon)

### Output Files

All outputs are saved to `temp/tutorials/editing/` for inspection.